# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
import os, json, sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)


def find_pack_root():
    """Pack root = folder with database/flasheats.db AND the Class 6/7 files (the updated instructor pack).

    Search order: $FLASHEATS_PACK, Challenges/flasheats-classroom-pack (created by get_pack.sh),
    the working directory and its parents, ~/flasheats-classroom-pack, then the Colab location.
    """
    candidates = [Path(os.environ["FLASHEATS_PACK"])] if os.environ.get("FLASHEATS_PACK") else []
    candidates += [Path.cwd() / "flasheats-classroom-pack", Path.cwd() / "Challenges" / "flasheats-classroom-pack",
                   Path.cwd(), *Path.cwd().parents, Path.home() / "flasheats-classroom-pack",
                   Path("/content/FlashEats_Classroom_Pack")]
    for folder in candidates:
        if (folder / "database" / "flasheats.db").exists() and (folder / "data" / "order_outcomes.csv").exists():
            return folder
    raise FileNotFoundError("FlashEats pack not found. Run `bash Challenges/get_pack.sh` or set FLASHEATS_PACK.")


BASE = find_pack_root()
OUTPUT = Path.cwd() / "output" if (Path.cwd() / "get_pack.sh").exists() else Path.cwd() / "Challenges" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Pack:", os.path.relpath(BASE))

import numpy as np
import matplotlib.pyplot as plt
DB_PATH = BASE / "database" / "flasheats.db"
print("Database exists:", DB_PATH.exists())

Pack: flasheats-classroom-pack


Database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order |  |  |  |
| Delivered orders have completion time |  |  |  |
| Promised ETA is valid |  |  |  |
| Event chronology is valid |  |  |  |
| “Late” has an agreed definition |  |  |  |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

### Our validation contract

| Business assumption | Data expectation | How we test it | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` unique in `orders` | count rows vs distinct IDs; inspect duplicates | **High**: inflates the denominator; conflicting duplicates are ambiguous |
| Delivered orders have completion time | every *delivered* order has `actual_delivery_at` | count delivered with null completion time | **High**: unknowns can move the rate either way |
| Promised ETA is valid | `promised_eta >= created_at` | count violations | **High** for the orders concerned: they become the "worst" lates |
| Event chronology is valid | `created_at <= pickup_at <= actual_delivery_at` | count violations per pair | **Medium**: impossible durations |
| "Late" has an agreed definition | one owner-approved rule (threshold, population) | read `client_metric_definitions.json` | **Critical**: no owner means no publishable number |

In [3]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))

# TODO:
# - establish business grain
# - check delivered orders with missing completion time
# - test promised_eta >= created_at
# - test pickup_at <= actual_delivery_at
# - mark which assumptions need stakeholder clarification

Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


In [4]:
o = orders.copy()
for col in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    o[col] = pd.to_datetime(o[col], format="mixed", errors="coerce")
o["status"] = o["final_status"].str.strip().str.lower()

dup_ids = o[o["order_id"].duplicated(keep=False)].sort_values("order_id")
conflicting = dup_ids.groupby("order_id").nunique().gt(1).any(axis=1).sum()
delivered_rows = o.drop_duplicates("order_id")
delivered_rows = delivered_rows[delivered_rows["status"] == "delivered"]

checks = pd.DataFrame([
    ("business grain", "order_id unique", f"{len(o)} rows / {o['order_id'].nunique()} ids; {conflicting} duplicate ids conflict (traffic_bucket)", "FAIL"),
    ("completion time", "delivered => actual_delivery_at", f"{int(delivered_rows['actual_delivery_at'].isna().sum())} of {len(delivered_rows)} delivered orders have none", "WARN"),
    ("promised ETA", "promised_eta >= created_at", f"{int((o['promised_eta'] < o['created_at']).sum())} orders promise delivery before the order existed", "WARN"),
    ("chronology", "pickup <= delivery", f"{int((o['actual_delivery_at'] < o['pickup_at']).sum())} orders delivered before pickup; "
                                          f"{int((o['pickup_at'] < o['created_at']).sum())} picked up before creation", "WARN"),
    ("definition of late", "one owner-approved rule", "4 stakeholders, 4 definitions, no documented KPI owner", "UNKNOWN"),
], columns=["assumption", "expectation", "evidence", "status"])
display(dup_ids[["order_id", "created_at", "actual_delivery_at", "final_status", "traffic_bucket"]])
display(checks)

,order_id,created_at,actual_delivery_at,final_status,traffic_bucket
119,O00120,2026-08-13 18:15:00,2026-08-13 19:34:08.431508,delivered,severe
1600,O00120,2026-08-13 18:15:00,2026-08-13 19:34:08.431508,delivered,medium
722,O00723,2026-08-05 21:04:00,2026-08-05 22:04:30.928322,delivered,medium
1601,O00723,2026-08-05 21:04:00,2026-08-05 22:04:30.928322,delivered,high
1301,O01302,2026-08-26 21:07:00,2026-08-26 22:20:27.428561,delivered,medium
1602,O01302,2026-08-26 21:07:00,2026-08-26 22:20:27.428561,delivered,high


,assumption,expectation,evidence,status
0,business grain,order_id unique,1603 rows / 1600 ids; 3 duplicate ids conflict (traffic_bucket),FAIL
1,completion time,delivered => actual_delivery_at,37 of 1532 delivered orders have none,WARN
2,promised ETA,promised_eta >= created_at,4 orders promise delivery before the order existed,WARN
3,chronology,pickup <= delivery,5 orders delivered before pickup; 0 picked up before creation,WARN
4,definition of late,one owner-approved rule,"4 stakeholders, 4 definitions, no documented KPI owner",UNKNOWN


**What could make "56%" misleading:**
- **3 conflicting duplicate orders.** They agree on timestamps but disagree on `traffic_bucket`, so the grain FAILs for any traffic breakdown.
- **37 delivered orders with no completion time.** "56%" silently excludes them.
- **4 impossible promises** (ETA before creation). They are the four most "late" orders.
- **5 deliveries timestamped before pickup.**
- **Most important: nobody owns the definition of "late".**

The number is *arithmetically* reproducible (Challenge 2). The problem is that it is *not yet a defined metric*.

# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

In [5]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60

# TODO: calculate the late rate using different definitions.

In [6]:
valid = analysis[(analysis["final_status"].str.strip().str.lower() == "delivered") & analysis["actual_delivery_at"].notna()]
delivered_all = analysis[analysis["final_status"].str.strip().str.lower() == "delivered"]
cancelled = int((analysis["final_status"].str.strip().str.lower() == "cancelled").sum())
unknown = int(delivered_all["actual_delivery_at"].isna().sum())
late_any = int((valid["delay_min"] > 0).sum())

hist = orders[(orders["final_status"] == "delivered") & orders["actual_delivery_at"].notna()].copy()
hist_delay = (pd.to_datetime(hist["actual_delivery_at"], format="mixed") - pd.to_datetime(hist["promised_eta"], format="mixed")).dt.total_seconds() / 60

definitions = pd.DataFrame([
    ("VP Ops: any delay > 0 min (delivered, known time)", late_any, len(valid), "Every broken promise counts"),
    ("Support: delay > 10 min", int((valid["delay_min"] > 10).sum()), len(valid), "Only delays customers feel strongly"),
    ("Data team: historical dashboard (exact 'delivered', no de-dup)", int((hist_delay > 0).sum()), len(hist), "Reproduces the published 56%, with its grain errors"),
    ("Finance: exclude cancelled/refunded (same as VP Ops here)", late_any, len(valid), "Cancelled orders already excluded; refunds are not in orders"),
    ("Sensitivity: every unknown-time delivery counted late", late_any + unknown, len(delivered_all), "Worst case for the 37 unknowns"),
    ("Sensitivity: every unknown-time delivery counted on time", late_any, len(delivered_all), "Best case for the 37 unknowns"),
    ("Sensitivity: cancelled counted as failures", late_any + cancelled, len(valid) + cancelled, "Customer-experience view of failure"),
], columns=["definition", "late", "denominator", "business meaning"])
definitions["late_rate_%"] = (100 * definitions["late"] / definitions["denominator"]).round(1)
display(definitions[["definition", "late", "denominator", "late_rate_%", "business meaning"]])

,definition,late,denominator,late_rate_%,business meaning
0,"VP Ops: any delay > 0 min (delivered, known time)",843,1495,56.4,Every broken promise counts
1,Support: delay > 10 min,349,1495,23.3,Only delays customers feel strongly
2,"Data team: historical dashboard (exact 'delivered', no de-dup)",841,1493,56.3,"Reproduces the published 56%, with its grain errors"
3,Finance: exclude cancelled/refunded (same as VP Ops here),843,1495,56.4,Cancelled orders already excluded; refunds are not in orders
4,Sensitivity: every unknown-time delivery counted late,880,1532,57.4,Worst case for the 37 unknowns
5,Sensitivity: every unknown-time delivery counted on time,843,1532,55.0,Best case for the 37 unknowns
6,Sensitivity: cancelled counted as failures,911,1563,58.3,Customer-experience view of failure


| Definition | Late rate | Business meaning |
|---|---:|---|
| Any delay > 0 min (VP Operations) | **56.4%** (843 / 1,495) | every broken promise |
| Delay > 10 min (Support Lead) | **23.3%** (349 / 1,495) | delays customers feel strongly |
| Historical dashboard (Data Team) | **56.3%** (841 / 1,493) | reproduces "56%", but with grain errors (3 duplicates in, 5 "Delivered" out) |
| Unknown-time orders counted as late / on time | **57.4% / 55.0%** | the range the 37 missing timestamps allow |

**Which should leadership publish?** Not "56%" on its own.
- **As a measure of broken promises,** the VP Operations definition, computed on the *corrected* grain: 56.4%, stated with its population and the 55.0–57.4% uncertainty band.
- **As a customer-pain measure,** the >10 min rate (23.3%) alongside it.

**Who must own the decision?** The metric definitions note says no canonical KPI owner is documented.
- **The VP Operations should be named owner of the delivery KPI.** Operations answers for the promise.
- **Finance and Support must sign off on the exclusions and the threshold.**
- **The Data Team implements it but should not decide it.**

# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [7]:
for col in ["final_status", "traffic_bucket"]:
    print("\n", col)
    print(orders[col].value_counts(dropna=False))

print("\nRestaurant status")
print(restaurant_status["status"].value_counts(dropna=False))

print("\nSupport category")
print(tickets["category"].value_counts(dropna=False))


 final_status
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

 traffic_bucket
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

Restaurant status
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1
Name: count, dtype: int64

Support category
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64


In [8]:
def show_values(label, s):
    vc = s.value_counts(dropna=False)
    norm = s.astype("string").str.strip().str.lower()
    print(f"\n{label}: {len(vc)} raw values -> {norm.nunique(dropna=False)} after trim+lowercase")
    display(pd.DataFrame({"raw_value": [repr(v) for v in vc.index], "count": vc.values}))

show_values("orders.final_status", orders["final_status"])
show_values("orders.traffic_bucket", orders["traffic_bucket"])
show_values("restaurant_status.status", restaurant_status["status"])
show_values("support_tickets.category", tickets["category"])


orders.final_status: 3 raw values -> 2 after trim+lowercase


,raw_value,count
0,'delivered',1530
1,'cancelled',68
2,'Delivered',5



orders.traffic_bucket: 5 raw values -> 4 after trim+lowercase


,raw_value,count
0,'medium',635
1,'high',448
2,'low',381
3,'severe',136
4,'HIGH',3



restaurant_status.status: 8 raw values -> 5 after trim+lowercase


,raw_value,count
0,'preparing',171
1,'handed_off',168
2,'ready',157
3,'ready ',2
4,'READY',1
5,'Ready',1
6,'handoff',1
7,'unknown',1



support_tickets.category: 9 raw values -> 8 after trim+lowercase


,raw_value,count
0,'late_delivery',38
1,'eta_changed',37
2,'restaurant_delay',37
3,'ready_but_waiting',33
4,'status_mismatch',29
5,'driver_not_moving',25
6,'Late Delivery',1
7,'late_delivery ',1
8,'ETA issue',1


| Field | Representation differences (safe to normalise: trim + lowercase) | Semantic questions (need an owner) |
|---|---|---|
| `final_status` | `"Delivered"` (5) = `"delivered"` | none: only delivered / cancelled exist |
| `traffic_bucket` | `"HIGH"` (3) = `"high"` | Who assigns the bucket, and when (at order time, or after the trip)? The 3 conflicting duplicate orders prove it can change. |
| restaurant `status` | `"READY"`, `"Ready"`, `"ready "` = `ready` | **`"handoff"` vs `"handed_off"`**: probably the same step, but a status is a workflow state, so the **restaurant-ops owner confirms** before merging. **`"unknown"` (1)**: keep as unknown, never guess. |
| ticket `category` | `"Late Delivery"`, `"late_delivery "` = `late_delivery` | **`"ETA issue"`**: is it `eta_changed` or `late_delivery`? **Support owns that mapping.** |

**Rule applied:** case and whitespace are normalised in code, because they are representation. Every *meaning* merge goes into an owner-approved mapping table and is never hard-coded by instinct.

# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [9]:
# Example:
# orders["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean()

# TODO: calculate mapping coverage for all relationships.

In [10]:
oid = orders.drop_duplicates("order_id")

def coverage(relationship, child, parent, risk):
    keys = child.dropna()
    return {"relationship": relationship, "rows": len(child), "null keys": int(child.isna().sum()),
            "coverage % (non-null)": round(100 * keys.isin(parent).mean(), 2),
            "coverage % (all rows)": round(100 * child.isin(parent).mean(), 2), "risk": risk}

cov = pd.DataFrame([
    coverage("orders.restaurant_id -> restaurants", oid["restaurant_id"], restaurants["restaurant_id"],
             "3 orders cannot be attributed to a restaurant (restaurant ranking)"),
    coverage("orders.driver_id -> drivers", oid["driver_id"], drivers["driver_id"],
             "3 other orders: no driver attribution"),
    coverage("tickets.order_id -> orders", tickets["order_id"], oid["order_id"],
             "3 tickets float free: complaint volume OK, ticket-to-order analysis loses them"),
    coverage("restaurant_status.order_id -> orders", restaurant_status["order_id"], oid["order_id"],
             "status exists for 500 orders only; 2 orders have two status rows"),
])
cov["status"] = np.where(cov["coverage % (all rows)"] == 100, "PASS", "WARN")
display(cov)
print("orders missing restaurant/driver:", oid[oid["restaurant_id"].isna()]["order_id"].tolist())
print("restaurant_status rows whose restaurant_id disagrees with the order's:",
      int((restaurant_status.merge(oid[["order_id", "restaurant_id"]], on="order_id", suffixes=("", "_order"))
           .pipe(lambda x: x["restaurant_id"] != x["restaurant_id_order"])).sum()))
print("orders with a restaurant_status row:", restaurant_status["order_id"].nunique(), "of", oid["order_id"].nunique())

,relationship,rows,null keys,coverage % (non-null),coverage % (all rows),risk,status
0,orders.restaurant_id -> restaurants,1600,3,100.0,99.81,3 orders cannot be attributed to a restaurant (restaurant ranking),WARN
1,orders.driver_id -> drivers,1600,3,100.0,99.81,3 other orders: no driver attribution,WARN
2,tickets.order_id -> orders,202,3,100.0,98.51,"3 tickets float free: complaint volume OK, ticket-to-order analysis loses them",WARN
3,restaurant_status.order_id -> orders,502,0,100.0,100.00,status exists for 500 orders only; 2 orders have two status rows,PASS


orders missing restaurant/driver: ['O00401', 'O00402', 'O00403']
restaurant_status rows whose restaurant_id disagrees with the order's: 0
orders with a restaurant_status row: 500 of 1600


| Relationship | Coverage % (all rows) | Status | Risk |
|---|---:|---|---|
| orders → restaurants | 99.81% (3 null keys: O00401–O00403; every non-null key resolves) | **WARN** | 3 orders drop out of any restaurant ranking |
| orders → drivers | 99.81% (3 *other* orders: O00404–O00406) | **WARN** | no driver attribution for them |
| tickets → orders | 98.51% (3 tickets with no order) | **WARN** | complaint counts fine; ticket-to-lateness analysis loses 3 |
| restaurant_status → orders | 100% | **PASS** (key integrity) | but only **500 of 1,600** orders have a status record at all (502 rows; 2 orders have two) |

**Is 1% unmapped acceptable?** It depends on the decision:
- **Company-wide late rate:** fine. 3 orders cannot move 56.4% measurably.
- **Ranking a single restaurant or driver for accountability:** *not* fine if the missing orders belong to the one being judged. Those 6 records must be fixed or excluded explicitly before any individual is named.

# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [11]:
# TODO:
# Merge restaurant_status with orders on order_id
# Parse timestamps
# Compare last_updated_at with order lifecycle timestamps

In [12]:
rs = restaurant_status.merge(oid[["order_id", "created_at", "pickup_at", "actual_delivery_at"]], on="order_id")
for col in ["last_updated_at", "created_at", "pickup_at", "actual_delivery_at"]:
    rs[col] = pd.to_datetime(rs[col], format="mixed", errors="coerce")
rs["status_norm"] = rs["status"].str.strip().str.lower()
rs["update_vs_pickup_min"] = (rs["last_updated_at"] - rs["pickup_at"]).dt.total_seconds() / 60
rs["update_vs_created_min"] = (rs["last_updated_at"] - rs["created_at"]).dt.total_seconds() / 60

fresh = pd.Series({
    "status rows": len(rs),
    "timestamps rounded to the whole minute": f"{100 * (rs['last_updated_at'].dt.second == 0).mean():.0f}%",
    "last update BEFORE the order existed": int((rs["update_vs_created_min"] < 0).sum()),
    "last status still 'preparing' although pickup happened": int((rs["status_norm"] == "preparing").sum()),
    "'ready' recorded only AFTER pickup": int(((rs["status_norm"] == "ready") & (rs["update_vs_pickup_min"] > 0)).sum()),
    "any status recorded after pickup": int((rs["update_vs_pickup_min"] > 0).sum()),
    "median minutes between last update and pickup": round(rs["update_vs_pickup_min"].median(), 1),
})
display(fresh.to_frame("value"))
display(rs.groupby("status_norm")["update_vs_pickup_min"].describe()[["count", "25%", "50%", "75%", "min", "max"]].round(1))

,value
status rows,502
timestamps rounded to the whole minute,100%
last update BEFORE the order existed,6
last status still 'preparing' although pickup happened,171
'ready' recorded only AFTER pickup,60
any status recorded after pickup,166
median minutes between last update and pickup,-6.6


,count,25%,50%,75%,min,max
status_norm,,,,,,
handed_off,168.0,-15.5,-6.4,3.0,-368.5,17.6
handoff,1.0,8.1,8.1,8.1,8.1,8.1
preparing,171.0,-16.6,-7.3,1.8,-380.4,23.7
ready,161.0,-17.2,-5.4,5.3,-385.8,25.4
unknown,1.0,-3.2,-3.2,-3.2,-3.2,-3.2


**Freshness is a question about the SLA for each use case:**

| Use case | Needs | Verdict |
|---|---|---|
| Weekly analytics (how often restaurants are "ready" before pickup) | an approximately correct status within hours | **WARN**: usable in aggregate. Minute rounding and some lag are acceptable; the 6 updates that pre-date the order must be excluded. |
| Live customer ETA | a status within about 1–2 minutes of reality | **FAIL**: every timestamp is minute-rounded. Updates are manual (restaurant tablet). **60 orders show "ready" only after the driver already picked up**, and 171 orders never moved past "preparing" before pickup. |
| Restaurant accountability ("food not ready") | an auditable, timely event per order | **FAIL**: only 500 of 1,600 orders have any status, the last-write-wins table keeps no history, and stale or late statuses would blame restaurants unfairly. |

**The same record can be fine for one use and unsafe for another.** That is why freshness is an SLA question, not a single data property.

# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain |  |  |  |
| Timestamp chronology |  |  |  |
| KPI definition |  |  |  |
| Category semantics |  |  |  |
| Cross-source mapping |  |  |  |
| Freshness |  |  |  |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

In [13]:
validation_report = {
    "business_grain": "FAIL - 3 conflicting duplicate order_ids (timestamps agree, traffic_bucket differs); must dedupe + resolve",
    "timestamp_chronology": "WARN - 4 ETAs before creation, 5 deliveries before pickup, 37 delivered orders with no completion time",
    "kpi_definition": "UNKNOWN - four stakeholder definitions, no documented owner (56.4% vs 23.3% depending on threshold)",
    "category_semantics": "WARN - case/space variants safe to normalise; 'handoff' vs 'handed_off' and 'ETA issue' need owners",
    "cross_source_mapping": "WARN - 3 orders without restaurant, 3 others without driver, 3 tickets without order; fine for the company rate, not for rankings",
    "freshness": "WARN/FAIL - WARN for weekly analytics, FAIL for live ETA and restaurant accountability",
    "publish_56_percent": "NO - publish only as a defined, owned metric with population and uncertainty stated",
}
gate = pd.DataFrame([(k, v.split(" - ")[0], v.split(" - ", 1)[1]) for k, v in validation_report.items()],
                    columns=["check", "status", "evidence / action"])
display(gate)

,check,status,evidence / action
0,business_grain,FAIL,"3 conflicting duplicate order_ids (timestamps agree, traffic_bucket differs); must dedupe + resolve"
1,timestamp_chronology,WARN,"4 ETAs before creation, 5 deliveries before pickup, 37 delivered orders with no completion time"
2,kpi_definition,UNKNOWN,"four stakeholder definitions, no documented owner (56.4% vs 23.3% depending on threshold)"
3,category_semantics,WARN,case/space variants safe to normalise; 'handoff' vs 'handed_off' and 'ETA issue' need owners
4,cross_source_mapping,WARN,"3 orders without restaurant, 3 others without driver, 3 tickets without order; fine for the company rate, not for rankings"
5,freshness,WARN/FAIL,"WARN for weekly analytics, FAIL for live ETA and restaurant accountability"
6,publish_56_percent,NO,"publish only as a defined, owned metric with population and uncertainty stated"


| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | **FAIL** | 1,603 rows / 1,600 IDs; the 3 duplicates conflict on `traffic_bucket` | De-duplicate; the source-system owner decides which traffic value is true; add a uniqueness constraint |
| Timestamp chronology | **WARN** | 4 ETAs before creation, 5 deliveries before pickup, 37 delivered orders with no completion time | Quarantine the 9 impossible orders; backfill the 37 from the driver app after sign-off |
| KPI definition | **UNKNOWN** | 4 definitions; 23.3%–58.3% depending on the rule; no owner | Name the owner (VP Ops); record the definition, population and threshold |
| Category semantics | **WARN** | case/space variants; `handoff`/`handed_off`; `ETA issue` | Normalise representation; owner-approved mapping table for meaning |
| Cross-source mapping | **WARN** | 3 orders without restaurant, 3 others without driver; 3 orphan tickets | Fine for the company rate; fix before any individual ranking |
| Freshness | **WARN / FAIL** | minute-rounded manual statuses; "ready" after pickup; only 500 orders covered | OK for weekly trends; not for live ETA or accountability |

**Should leadership publish "Late Delivery Rate = 56%" today? No, not in that form.**
- The arithmetic is reproducible, but the grain FAILs and the definition is UNKNOWN.
- Leadership may state: *"56.4% of delivered orders with a recorded delivery time arrived after the promised ETA (843 / 1,495; 55.0–57.4% allowing for 37 orders with no delivery time). Pending an agreed KPI definition."*

**Before it becomes publishable:**
1. The VP Operations is named KPI owner and signs a definition (threshold, population, handling of cancellations and missing times) with Finance and Support.
2. `order_id` uniqueness is enforced and the 3 conflicts are resolved.
3. The 9 chronology-broken orders are quarantined and the 37 missing delivery times are backfilled or explicitly excluded.
4. The published figure carries its population and its uncertainty band.

# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**